# Data Engineering Lab Midsem 2025

You are given a dataset for classification with m features, n samples and the target attribute (k possible values). Assume that all the features are discrete. You need to write python codes for the given three problems.

Submit this as a single python notebook file, with three clearly defined sections for the three problems. The name of your notebook file should be: `midsem-roll.ipynb`. A sample data is given for developing your program, where ‘-‘ means a missing value and the first row gives you the feature names.

Print your output on screen in the following format.

Problem 1: sample id, feature name, value (one line for one record)

Problem 2: The feature name (one line for one feature)

Problem 3: Feature A, Feature B

In [1]:
import numpy as np
import pandas as pd

In [2]:
df = pd.read_excel('midsem-sample-data-2025.xlsx', index_col=0)
display(df)

,Outlook,Temp,Humidity,Wind,Play?
ID,,,,,
1,Sunny,Hot,High,Weak,No
2,Sunny,Hot,High,Strong,No
3,Overcast,Hot,High,Weak,Yes
4,Rain,Mild,High,Weak,Yes
5,Rain,Cool,-,Weak,Yes
6,Rain,Cool,Normal,Strong,No
7,Overcast,Cool,Normal,Strong,Yes
8,Sunny,Mild,High,Weak,No
9,Sunny,-,Normal,Weak,Yes


### 1. Fill out missing values of features by taking the most likely value of the feature for the class.

In [3]:
def compute_likelihood(df, feature, target):
    """
    Compute p(x | y) for each label in x and each class in y 
    where x = df[feature] and y = df[target]
    """
    likelihood = {}
    counts = pd.crosstab(df[feature], df[target])
    class_counts = df[target].value_counts()
    
    for label in df[feature].unique():
        if label == '-':
            continue
        
        likelihood[label] = {}
        for class_label in class_counts.index:
            # p(x = xi| y = k) = p(x = xi, y = k) / p(y = k)
            likelihood[label][class_label] = np.round(
                counts.loc[label].get(class_label, 0) / class_counts[class_label], 2
            )
    
    return likelihood

In [4]:
def fill_missing_values(df, target):
    """
    Fill the missing values of features by taking the most likely value of the feature for that class
    """
    missing_dict = []
    
    for feature in df.columns:
        if feature == target:
            continue
        
        missing_rows = df[df[feature] == '-']
        if len(missing_rows) == 0:
            continue
        
        likelihood = compute_likelihood(df, feature, target)       
        range_idx = df[df[feature] == '-'].index
        for idx in range_idx:
            class_label = df.loc[idx, target]
            most_likely_label = '-'
            max_likelihood = 0.0            
            for label in likelihood:
                if likelihood[label][class_label] > max_likelihood:
                    most_likely_label = label
                    max_likelihood = likelihood[label][class_label]
                    
            df.loc[idx, feature] = most_likely_label
            missing_dict.append({'Sample ID': idx, 'Feature': feature, 'Value': most_likely_label})
    
    return pd.DataFrame(missing_dict)

In [5]:
display(fill_missing_values(df, 'Play?'))

,Sample ID,Feature,Value
0,9,Temp,Mild
1,5,Humidity,Normal


### 2. Rank the features based on their potential values (in descending order). 

The potential value of a feature A is computed as

𝑃𝑜𝑡𝑒𝑛𝑡𝑖𝑎𝑙(𝐴) = 𝑅(𝑆) − Σ|𝑆𝑣|/|𝑆|×𝑅(𝑆𝑣) 𝑣∈𝑉𝑎𝑙𝑢𝑒𝑠(𝐴)

where 𝑅(𝑆)= − Σ𝑝𝑖log2𝑝𝑖, 𝑖=1 to c, 

𝑝𝑖 denotes the probability of a sample in S to be in class 𝑖 and c denotes the total number of classes in the data. 𝑆𝑣 denotes the set of samples in S that has value v for the attribute A.

In [6]:
def entropy(df, target):
    p = df[target].value_counts(normalize = True)
    r = - np.sum(p * np.log2(p))
    return r

In [7]:
def potential(df, feature, target):
    """
    Compute potential value of the feature
    """
    r_feature = entropy(df, target)
    weights = df[feature].value_counts(normalize=True)
    r_feature_labels = [entropy(df[df[feature] == label], target) for label in weights.index]
    info_gain = r_feature - np.sum(weights * r_feature_labels)
    return info_gain

In [8]:
def rank_features(df, target):
    """
    Rank the features based on their potential values
    """
    potentials = {}
    for feature in df.columns:
        if feature != target:
            potentials[feature] = potential(df, feature, target)
    
    rankings = sorted(potentials.items(), key=lambda x: x[1], reverse=True)
    return pd.DataFrame(rankings, columns=['Feature', 'Potential'])

In [9]:
display(rank_features(df, 'Play?'))

,Feature,Potential
0,Outlook,0.246750
1,Humidity,0.151836
2,Wind,0.048127
3,Temp,0.026234


### 3. Find out the most potential pair of features based on the following measure:

𝑃𝑜𝑡𝑒𝑛𝑡𝑖𝑎𝑙(𝐴,𝐵)=𝑅(𝑆) −Σ|𝑆𝑢,𝑣|/|𝑆|×𝑅(𝑆𝑢,𝑣)

(𝑢,𝑣)∈(𝑉𝑎𝑙𝑢𝑒𝑠(𝐴)×𝑉𝑎𝑙𝑢𝑒𝑠(𝐵))

In [10]:
def pair_potential(df, feature1, feature2, target):
    """
    Compute potential value of the given pair of features
    """
    r_feature = entropy(df, target)
    weights = pd.crosstab(df[feature1], df[feature2], normalize=True)
    r_feature_labels = []
    
    for label1 in weights.index:
        for label2 in weights.columns:
            subset = df[(df[feature1] == label1) & (df[feature2] == label2)]
            r_feature_labels.append(entropy(subset, target))
            
    info_gain = r_feature - np.sum(weights.values.flatten() * r_feature_labels)
    return info_gain

In [11]:
def most_potential_pair(df, target):
    """
    Fetch the feature pair with best potential value
    """
    max_potential = 0.0
    best_pair = ('', '')
    features = df.columns
    
    for i in range(1, len(features)):
        if features[i] == target:
            continue
        
        for j in range(i):
            if features[j] == target:
                continue
            
            curr_potential = pair_potential(df, features[j], features[i], target)
            if curr_potential > max_potential:
                best_pair = (features[j], features[i])
                max_potential = curr_potential
                
    return best_pair

In [12]:
most_potential_pair(df, 'Play?')

('Outlook', 'Humidity')

---
End of the assignment